In [1]:
%pip install --force-reinstall --no-deps git+https://github.com/chrisjcameron/TexSoup.git@mixed-args

  Cloning https://github.com/chrisjcameron/TexSoup.git (to revision mixed-args) to /var/tmp/pip-req-build-gw9hpcec
  Running command git clone --filter=blob:none --quiet https://github.com/chrisjcameron/TexSoup.git /var/tmp/pip-req-build-gw9hpcec
  Running command git checkout -b mixed-args --track origin/mixed-args
  Switched to a new branch 'mixed-args'
  Branch 'mixed-args' set up to track remote branch 'mixed-args' from 'origin'.
  Resolved https://github.com/chrisjcameron/TexSoup.git to commit a5f0d18d89bf0df21f0b47d4bea83dbdabfa93e9
  Preparing metadata (setup.py) ... done
  Created wheel for TexSoup: filename=texsoup-0.3.1-py3-none-any.whl size=32780 sha256=07929228a207e062f92eb82a54b604ecfbec9e232eb718cd2c401588c6271dae
  Stored in directory: /var/tmp/pip-ephem-wheel-cache-nez3d9s2/wheels/92/51/24/650922238315330304591a8b578bb8041933e0646456703bfa
Successfully built TexSoup
  Attempting uninstall: TexSoup
    Found existing installation: TexSoup 0.3.1
    Uninstalling TexSoup-0

In [1]:
#%pip install -U regex

In [61]:
import tarfile
import zipfile
import io
import os
import glob
import time
import math
import pickle
import itertools as itr
import collections as coll
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm
from pytictoc import TicToc
import json
import datetime

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

from google.cloud import storage
from google.resumable_media.common import InvalidResponse
from google.cloud.exceptions import ClientError
from google.api_core.exceptions import GoogleAPIError, NotFound, Forbidden


PROJECT_ID = "arxiv-development"
PRD_PROJECT = 'arxiv-production'
PRD_BUCKET_LOC = 'arxiv-production-data' 

from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode, LatexCharsNode
from pylatexenc.latex2text import LatexNodes2Text


In [62]:
os.chdir("/home/jupyter/metadata-vertexai/")  # this needs to be the folder where notebook lives
import importlib
import phase_one_json as phase_one


In [63]:
#ror_rebuilt = phase_one.rorFinder(RECREATE_INDEX=True)

In [64]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

In [65]:
def safe_divide(num, denom):
    return num / denom if denom != 0 else 0.0

In [66]:
import TexSoup as TS
from TexSoup.tokens import MATH_ENV_NAMES

In [8]:
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()

new_def_pat = re.compile(new_def_str)

In [6]:
foo = np.array([list("abc")])
foo.shape

(1, 3)

In [54]:
import pickle
time_code = "2025-05-12"
save_name = "2024_db_json"
sample_size = "all"
batch_size = 20
parallel_workers = 30 #8
thread_workers = 10

try:
    objects = []
    grp_num = 0
    with open(f"checkpoints/{save_name}_{sample_size}_{time_code}.pkl", 'rb') as cp_fp:
        while True:
            try:
                obj = pickle.load(cp_fp)
                objects.append((grp_num, obj))
            except EOFError:
                break
except FileNotFoundError as e:
    pass

known_ids = []
known_res = []

for grp_num, obj in objects:
    known_ids.extend(x[0] for x in obj)
    known_res.extend(obj)
    
known_res[-20:]

[(np.str_('2412.10047v2'), 'Microsoft', '', '00d0nc645'),
 (np.str_('2412.10047v2'), 'Peking University', '', '02v51f717'),
 (np.str_('2412.10047v2'), 'Zhejiang University', '', '00a2xv884'),
 (np.str_('2412.10047v2'),
  'Eindhoven University of Technology',
  '',
  '02c2kyt77'),
 (np.str_('2404.15650v3'), 'Seoul National University', '', '04h9pn542'),
 (np.str_('2404.15650v3'), 'LG AI Research', '', 'null'),
 (np.str_('2404.15650v3'), 'IPAI', '', '042688r20'),
 (np.str_('2404.15650v3'), 'NAVER Cloud', '', '04nzrnx83'),
 (np.str_('2404.15650v3'), 'University of Richmond', '', '03y71xh61'),
 (np.str_('2411.06939v3'),
  'Institute of High Energy Physics, Chinese Academy of Sciences',
  'Beijing',
  '03v8tnc06'),
 (np.str_('2411.06939v3'),
  'China Center of Advanced Science and Technology',
  'Beijing',
  '02egfyg20'),
 (np.str_('2411.06939v3'),
  'University of Chinese Academy of Sciences',
  'Beijing',
  '05qbk4x57'),
 (np.str_('2411.06939v3'),
  'Vanderbilt University Medical Center',

In [55]:
rerun_ids = set([str(x[0]) for x in known_res if x[1] in ('null', 'error')])
len(rerun_ids)

1117

In [56]:
arxid_itr = itr.groupby(sorted(known_res, key=lambda x: x[0]), key=lambda x: x[0])
no_result_idx = [str(arx_id) for arx_id, grp in arxid_itr if all(x[1] in ('null', 'error') for x in grp)]
no_result_idx[:10]
len(no_result_idx)

['2401.00016v2',
 '2401.00278v2',
 '2401.00837v1',
 '2401.00837v2',
 '2401.01452v1',
 '2401.01457v1',
 '2401.01668v1',
 '2401.01668v2',
 '2401.02242v1',
 '2401.02488v1']

1028

In [57]:
def get_outstanding_jobs_from_file(logfile):
    with open(logfile) as infile:
        lines = sorted(x.strip().split() for x in infile.readlines() if x.strip())
        cntr = coll.Counter(x[0] for x in lines)
        res = [k for k, cnt in cntr.items() if cnt < 2]
    return res

In [59]:
now = time.time()
check_ids = []
for logfile in glob.glob("logs/*.log"):
    lmt = os.path.getmtime(logfile)
    if now - lmt > 60*15:
        res = get_outstanding_jobs_from_file(logfile)
        check_ids.extend(res)
print(check_ids)

for arxid in check_ids:
    print(f"arx_id = '{arxid}'")

['2410.12880v2', '2410.12880v3']
arx_id = '2410.12880v2'
arx_id = '2410.12880v3'


Open after 15 mins:
```
arx_id = '2405.18008v1'
arx_id = '2403.04857v2'



```

15+ mins first epoch:

```
arx_id = '2410.12880v2'
arx_id = '2410.12880v3'
```




Open at end of the first OOM

```
arx_id = '2411.02966v1'
arx_id = '2403.00674v2'
arx_id = '2403.17095v1'
arx_id = '2404.03844v2'
arx_id = '2409.16341v1'
arx_id = '2406.00643v1'
arx_id = '2409.14982v1'
arx_id = '2404.09634v2'
arx_id = '2403.08111v1'
arx_id = '2404.18042v1'
arx_id = '2405.09083v1'
arx_id = '2406.12441v1'
arx_id = '2410.11688v1'
arx_id = '2401.00113v1'
arx_id = '2402.14681v2'
arx_id = '2402.16520v2'
arx_id = '2403.18736v1'
arx_id = '2404.03358v1'
arx_id = '2406.02038v1'
arx_id = '2407.01963v2'
arx_id = '2410.20849v1'
arx_id = '2411.06175v2'
arx_id = '2411.10337v1'
arx_id = '2404.03806v2'
arx_id = '2404.06829v1'
arx_id = '2404.17377v2'
arx_id = '2405.13491v1'
arx_id = '2407.10549v1'
arx_id = '2404.03533v1'
arx_id = '2401.13029v1'
arx_id = '2402.03448v3'
arx_id = '2402.15430v2'
arx_id = '2403.08987v2'
arx_id = '2403.13981v1'
arx_id = '2404.15327v1'
arx_id = '2406.09879v1'
arx_id = '2407.15527v2'
arx_id = '2408.06956v1'
arx_id = '2408.12786v1'
arx_id = '2412.18759v1'
```

In [53]:
tail = '''
2311.10270v5 start
2311.08987v1 start
2311.14186v1 start
2306.01021v1 start
2303.07569v2 start
2305.00312v3 start
2303.13827v2 start
2309.10262v2 start
2309.15957v5 start
2311.11770v1 start
2311.08987v1 stop
2307.08203v1 start
2303.13827v2 stop
2307.11797v5 start
2309.15957v5 stop
2304.05689v2 start
2307.11797v5 stop
2310.04217v2 start
2304.05689v2 stop
2303.14070v2 start
2303.14070v2 stop
2307.12602v2 start
2310.04217v2 stop
2306.13567v2 start
2302.10602v2 start
2311.14186v1 stop
2310.18275v1 start
2306.13567v2 stop
2311.14619v1 start
2302.10602v2 stop
'''.strip()

In [54]:
lines = sorted(x.split() for x in tail.splitlines())
cntr = coll.Counter(x[0] for x in lines)

In [55]:
cntr

Counter({'2302.10602v2': 2,
         '2303.13827v2': 2,
         '2303.14070v2': 2,
         '2304.05689v2': 2,
         '2306.13567v2': 2,
         '2307.11797v5': 2,
         '2309.15957v5': 2,
         '2310.04217v2': 2,
         '2311.08987v1': 2,
         '2311.14186v1': 2,
         '2303.07569v2': 1,
         '2305.00312v3': 1,
         '2306.01021v1': 1,
         '2307.08203v1': 1,
         '2307.12602v2': 1,
         '2309.10262v2': 1,
         '2310.18275v1': 1,
         '2311.10270v5': 1,
         '2311.11770v1': 1,
         '2311.14619v1': 1})

### Test Ids

```
'2301.08641v2': 1,
'2304.09870v2': 1,
'2309.01118v2': 1,
'2310.20374v3': 1,
'2303.11590v3': 1,
'2308.04512v1': 1,
'2312.05433v2': 1,
'2304.14219v4': 1,
'2307.05569v1': 1,
'2312.07121v1': 1,
'2312.14567v1': 1,
'2303.01063v2': 1,
'2308.04512v2': 1,
'2309.08117v3': 1,
'2302.07019v1': 1,
'2305.04720v2': 1,
'2306.03953v1': 1,
'2310.13041v1': 1,
'2310.19023v1': 1,
'2312.05433v1': 1,
'2309.05340v1': 1,
'2301.12994v2': 1,
'2302.12906v2': 1,
'2311.05999v2': 1,
'2311.17186v3': 1,
'2312.13411v1': 1,
'2310.20317v5': 1,
'2311.14636v3': 1,
'2305.04797v2': 1,
'2309.02994v1': 1,
```

Potentially:

```
         '2302.03142v1': 1,
         '2303.02844v1': 1,
         '2303.15746v1': 1,
         '2304.11351v3': 1,
         '2305.04797v2': 1,
         '2307.06155v5': 1,
         '2307.07457v1': 1,
         '2308.03822v1': 1,
         '2309.01218v2': 1,
         '2309.10165v2': 1,
         '2310.14128v1': 1,
         '2311.05966v5': 1,
         '2311.16440v2': 1,
         '2312.10691v1': 1,
         '2312.13175v2': 1,
         '2312.17319v1': 1
         '2303.07569v2': 1,
         '2305.00312v3': 1,
         '2306.01021v1': 1,
         '2307.08203v1': 1,
         '2307.12602v2': 1,
         '2309.10262v2': 1,
         '2310.18275v1': 1,
         '2311.10270v5': 1,
         '2311.11770v1': 1,
         '2311.14619v1': 1
```

## Test TexSoup

In [13]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2311_ids.csv")
test_ids_df.head()

,arx_id
0,2311.00001v1
1,2311.00002v1
2,2311.00003v4
3,2311.00004v3
4,2311.00005v1


In [ ]:
sample_size = 1000
batch_size = 20
parallel_workers = 8
thread_workers = 10
#import concurrent.futures
#import phase_one

def test_id(arx_id):
    yymm = arx_id.split(".")[0]
    paper_id = arx_id.split("v")[0]
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"

    res = None
    try:
        tar_bytes = phase_one.bytes_from_tarpath(tar_path)
        candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
        source_text = phase_one.source_from_archive(tar_bytes, candidate_files[0])
        tsoup = TS.TexSoup(source_text, tolerance=0)
        return 1
    except KeyboardInterrupt:
        raise
    except:
        try:
            tsoup = TS.TexSoup(source_text, tolerance=1)
            return 2
        except :
            return 0
    return 0

res = []
for arx_id in tqdm(test_ids_df['arx_id'].iloc[0:100]):
    res.append(test_id(arx_id))
    
coll.Counter(res)

In [67]:
%%time
arx_id = '2308.03822v1'  ## long auth list, slow
#arx_id = '2302.03142v1'
#arx_id = '2303.02844v1'
#arx_id = '2303.15746v1'
#arx_id = '2304.11351v3'
#arx_id = '2305.04797v2'
#arx_id = '2307.06155v5'
#arx_id = '2307.07457v1'
#arx_id = '2309.01218v2'
#arx_id = '2309.10165v2'
#arx_id = '2310.14128v1'
#arx_id = '2311.05966v5'
#arx_id = '2311.16440v2'
#arx_id = '2312.10691v1'
#arx_id = '2312.13175v2'
#arx_id = '2312.17319v1'
#arx_id = '2303.07569v2'
#arx_id = '2305.00312v3'
#arx_id = '2306.01021v1'
#arx_id = '2307.08203v1'
#arx_id = '2307.12602v2'
#arx_id = '2309.10262v2'
#arx_id = '2310.18275v1'  ## slow gemini
#arx_id = '2311.10270v5'

arx_id = '2311.01592v3'
arx_id = '2410.12880v2'

phase_one.get_single_file_results(arx_id, verbose=True, vverbose=True)

Processing ftp/arxiv/papers/2410/2410.12880.tar.gz
	Processing ftp/arxiv/papers/2410/2410.12880.tar.gz, acl_latex.tex



KeyboardInterrupt



In [ ]:
%%time
arx_id = '2308.04512v1'
#arx_id = '2305.15927v4'
#arx_id = '2310.03838v2'  # No author tags, placed at end
#arx_id = '2306.01401v1'
arx_id = '2310.18275v1'  ## slow
arx_id = '2308.03822v1'  ## long auth list, slow
arx_id = '2410.12880v2'


yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"

try:
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
    tex_main = candidate_files[0]
except (NotFound, InvalidResponse):
    print("Got Invalid")
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.gz"
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    tex_main = include_dict = None
    candidate_files = [None]

inc_list = []
if include_dict: inc_list = include_dict.get(tex_main, [])
print(f"Got main: {tex_main} with {inc_list}")

total_res = []
for c_file in candidate_files:
    print(f"Starting: {c_file}")
    res_gen = phase_one.extract_pre_abstract_content(tar_bytes, tex_main=c_file, include_list=None, file_path=tar_path)
    res_list = list(res_gen)
    total_res.append(res_list)
    print(f"finished with {c_file}, got {len(res_list)}")

In [ ]:
len(total_res)
[(i, len(x)) for i,x in enumerate(total_res)]
total_res

In [70]:
arx_id = '2310.03838v1'  # actually fails #'symbols.tex'
#arx_id = '2306.01401v1'  # symbols_env.tex
#arx_id = '2306.14221v2'
#arx_id = '2310.16202v1'
arx_id = '2305.15927v4'
#arx_id = '2311.05999v2'
arx_id = '2310.18275v1'  ## slow
arx_id = '2410.12880v2'


def append_node_contents(focus_nodes, full_nodelist, result_list):
    for i,node in focus_nodes:
        result_list.append(node.latex_verbatim())
        try:
            idx_plus = 1
            group_streak=False
            while True:
                if idx_plus > 10:
                    break
                if (not group_streak) and (len(result_list) > 2):
                    break
                follow_node = full_nodelist[i+idx_plus]
                if isinstance(follow_node, LatexGroupNode):
                    result_list.append(follow_node.latex_verbatim())
                    group_streak = True
                elif isinstance(follow_node, LatexCharsNode):
                    if not str(follow_node.chars).isspace():
                        group_streak = False
                else:
                    group_streak = False
                idx_plus += 1
        except IndexError:
            pass

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"


tar_bytes = phase_one.bytes_from_tarpath(tar_path)
candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
tex_main = candidate_files[0]
include_list = None #[]
#if include_dict: include_list = include_dict.get(tex_main, [])

incl_res_gen_list = []
if include_list:
    for inc_file in include_list:
        incl_res_gen_list.append(extract_pre_abstract_content(tar_bytes, tex_main=inc_file, file_path=file_path))

#tex_main = "main.tex"  #
print(f"Got main: {tex_main}")
if include_dict:
    print(f"Got included_files: {include_dict.get('tex_main')}")

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""

source_text = phase_one.source_from_archive(tar_bytes, tex_main)


source_text = r"""
\begin{icmlauthorlist}
\icmlauthor{Vy Vo}{yyy,ccc}
\icmlauthor{Trung Le}{yyy}
\icmlauthor{Tung-Long Vuong}{yyy,vvv}
\icmlauthor{He Zhao}{ccc}
\icmlauthor{Edwin V. Bonilla}{ccc}
\icmlauthor{Dinh Phung}{yyy,vvv}
\end{icmlauthorlist}

\icmlaffiliation{yyy} {Monash University, Australia}
\icmlaffiliation{ccc}{CSIRO's Data61, Australia}
\icmlaffiliation{vvv}{VinAI Research, Vietnam}

\icmlcorrespondingauthor{Vy Vo}{v.vo@monash.edu}

""".strip()

# Remove LaTeX comments (lines starting with non-escaped %)
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()
new_def_v1 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*{[^{}]*}
""".strip()
new_def_v2 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{[^{}]*}
""".strip()
new_def_v3 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{(?:[^{}]*|{[^{}]*})*}\s*{(?:[^{}]*|{[^{}]*})*}
""".strip()

#old # \\newenvironment\{[^\}]+\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}
strip_env = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\[[^\]]*\])*(\{\s*(\s*(?>[^{}]+|\{(?3)\})*)+\}){1,3}
""".strip()
strip_provcmd = r"""
\\providecommand\{[^\}]+\}\s*(\[[^\]]*\])?\s*\{\s*((?>[^{}]+|\{(?:[^{}]*|(?1))\})*)\}
""".strip()
new_def_v1_pat = re.compile(new_def_v1, re.DOTALL, re.MULTILINE)
new_def_v2_pat = re.compile(new_def_v2, re.DOTALL, re.MULTILINE)
new_def_v3_pat = re.compile(new_def_v3, re.DOTALL, re.MULTILINE)
strip_env_pat  = re.compile(strip_env, re.DOTALL)
strip_provcmd_pat  = re.compile(strip_provcmd, re.DOTALL)
content = re.sub(r"(?<!\\)%.*", "", source_text)
content = strip_provcmd_pat.sub("\n", content)
content = strip_env_pat.sub("\n", content)
content = new_def_v3_pat.sub("\n", content)
#content = new_def_v1_pat.sub("\n", content)
#content = new_def_v2_pat.sub("\n", content)
#res_list = []

# try parsing latex:
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "affiliation", "affil", "affiliations",
    "address",
    "cmsinstitute", "icmlaffiliation",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []

lxwkr = LatexWalker(content, tolerant_parsing=True)
print("Starting get nodes main")
(nodelist, pos, len_) = lxwkr.get_latex_nodes(read_max_nodes=None) #789 #2122
print("Stopping get nodes main")
focus_nodes = [
  (i,node) for i,node in enumerate(nodelist)
  if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
]
if focus_nodes:
    append_node_contents(focus_nodes, nodelist, latex_extracted_institutions)
    # Get /textsuperscript contents if indicated
    # @todo: also get the $^[1]$Institution style indicators 
    if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
        sup_res = extract_texsuperscript(nodelist)
        latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            docnodelist = doc[0].nodelist
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(docnodelist)
              if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
            ]
            append_node_contents(focus_doc_nodes, docnodelist, latex_extracted_institutions)
            # Get /textsuperscript contents if indicated
            # @todo: also get the $^[1]$Institution style indicators 
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(docnodelist)
                latex_extracted_institutions.extend(sup_res)


len(nodelist)
#candidate_files

Got main: acl_latex.tex
Starting get nodes main
Stopping get nodes main


18

In [42]:
str(nodelist[4].chars).isspace()

True

In [43]:
nodelist[2:]

[LatexMacroNode(parsing_state=<parsing state 140094271611488>, pos=232, len=16, macroname='icmlaffiliation', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<parsing state 140094271611488>, pos=248, len=5, nodelist=[LatexCharsNode(parsing_state=<parsing state 140094271611488>, pos=249, len=3, chars='yyy')], delimiters=('{', '}')),
 LatexCharsNode(parsing_state=<parsing state 140094271611488>, pos=253, len=1, chars=' '),
 LatexGroupNode(parsing_state=<parsing state 140094271611488>, pos=254, len=30, nodelist=[LatexCharsNode(parsing_state=<parsing state 140094271611488>, pos=255, len=28, chars='Monash University, Australia')], delimiters=('{', '}')),
 LatexCharsNode(parsing_state=<parsing state 140094271611488>, pos=284, len=1, chars='\n'),
 LatexMacroNode(parsing_state=<parsing state 140094271611488>, pos=285, len=16, macroname='icmlaffiliation', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNod

In [29]:
focus_nodes


[(2,
  LatexMacroNode(parsing_state=<parsing state 140094931765712>, pos=232, len=16, macroname='icmlaffiliation', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')),
 (6,
  LatexMacroNode(parsing_state=<parsing state 140094931765712>, pos=284, len=16, macroname='icmlaffiliation', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')),
 (10,
  LatexMacroNode(parsing_state=<parsing state 140094931765712>, pos=333, len=16, macroname='icmlaffiliation', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''))]

## Test regex

In [98]:
test_string = r'''
\usepackage{bm}
\newenvironment{discussion}{\noindent  \newline \noindent \bf ************************** BEGIN Discussion\rm \\ \bf}{\rm \noindent \newline \noindent \bf ************************** END Discussion\rm\\}

\author{Walter Bridges, Johann Franke, and Johann Stumpenhusen}
\title{On the Proportion of Coprime Fractions in Number Fields}
\address{Department of Mathematics and Computer Science, Division of Mathematics, University of Cologne, Weyertal 86-90, 50931 Cologne, Germany}
\email{wbridges@uni-koeln.de}
\email{jfrank12@uni-koeln.de}
\email{jstumpen@math.uni-koeln.de}

\keywords{class group, density, Hecke $L$-function, Heegner points}

\begin{document}

\maketitle

\begin{abstract}
    We determine the asymptotic density of coprime fractions in those of the reduced fractions of number fields. When ordered by norms of denominators, we count a fraction as soon as it ``appears'' for the first time and no later.  The natural density of coprime fractions in the set of reduced fractions may then be computed using well-known facts about Hecke $L$-functions. 
 Furthermore, we draw some connections to the modular group and Heegner points.
\end{abstract}

'''.strip()

test_string = r'''
\newenvironment{enumerate-(a)}{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc]}{\end{enumerate}}
\newenvironment{enumerate-(a)-r}{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc,resume]}{\end{enumerate}}
\newenvironment{enumerate-(a)-5}{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc,start=5]}{\end{enumerate}}
'''.strip()

test_string = r'''
\newenvironment{theo}[1][]{%
\stepcounter{theo}%
\ifstrempty{#1}%
 {\mdfsetup{%
   frametitle={%
    \tikz[baseline=(current bounding box.east),outer sep=0pt]
    \node[anchor=east,rectangle,fill=PeriWinkle!80]
         {\strut Example~\thetheo};}}
 }%
{\mdfsetup{%
  frametitle={%
   \tikz[baseline=(current bounding box.east),outer sep=0pt]
   \node[anchor=east,rectangle,fill=purple!40]
        {\strut \footnotesize Example instances};}}%
 }%
\mdfsetup{innertopmargin=1pt,linecolor=purple!40,%
       linewidth=2pt,topline=true,
       frametitleaboveskip=\dimexpr-\ht\strutbox\relax,}
   \begin{mdframed}[]\relax%
}
{\end{mdframed}}

'''
initial_def = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(\s*\{\s*(\s*(?>(\\\\+|\\[{}]|[^{}\\])+|\{(?3)\})*|\\)+\}){1,3}
""".strip()


initial_def = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\}){1,3}
""".strip()

initial_def = r"""
\\newenvironment\s*\{[^\}]+\}(\s*\[[^\]]*\])*\s*(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})+
""".strip()

temp = '''
#(\s*\{\s*\\)               # start grp1 open brace

""".strip()
#    (?>(\\|[^{}\\]+|\{(?3)\}*))[^{}\\]+
#\}){1,3}                    # close grp 1 with close brace, repeat 1-3 times

'''

args = r"""
\s*(\[[^\]]*\])?\s*\{\s*((?>[^{}]+|\{(?:[^{}]*|(?1))\})*)\}""".strip()
rest = r"""
(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})+
""".strip()
test_pat = re.compile(initial_def, re.DOTALL, re.VERBOSE)
test_pat
m = test_pat.search(test_string)
m
#m.groupdict()
test_pat.sub('\n', test_string)

regex.Regex('\\\\newenvironment\\s*\\{[^\\}]+\\}(\\s*\\[[^\\]]*\\])*\\s*(?P<brgrp>\\s*\\{\\s*(?P<inner>(?>\\s+|\\\\\\\\+|\\\\[{}]|[^{}\\\\]+|\\\\)+|\\{(?P>inner)\\}+)+\\s*\\})+', flags=regex.S | regex.V0)

'\n\\newenvironment{theo}[1][]{%\n\\stepcounter{theo}%\n\\ifstrempty{#1}%\n {\\mdfsetup{%\n   frametitle={%\n    \\tikz[baseline=(current bounding box.east),outer sep=0pt]\n    \\node[anchor=east,rectangle,fill=PeriWinkle!80]\n         {\\strut Example~\\thetheo};}}\n }%\n{\\mdfsetup{%\n  frametitle={%\n   \\tikz[baseline=(current bounding box.east),outer sep=0pt]\n   \\node[anchor=east,rectangle,fill=purple!40]\n        {\\strut \\footnotesize Example instances};}}%\n }%\n\\mdfsetup{innertopmargin=1pt,linecolor=purple!40,%\n       linewidth=2pt,topline=true,\n       frametitleaboveskip=\\dimexpr-\\ht\\strutbox\\relax,}\n   \\begin{mdframed}[]\\relax%\n}\n{\\end{mdframed}}\n\n'

In [ ]:
m = regex.search(
    r'(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})', 
    r"{\begin{enumerate}[label={\upshape (\alph*)}, leftmargin=2pc]}"
)
m
m.groupdict()
re.DEFAULT_VERSION

In [68]:
print(content[99849:99849+300])




\lambda+1})=\bigcup\Set{\HOD^{L(V_{\lambda+1})}_{x}}{x\in V_{\lambda+1}}$} 

In addition, for all $x,y\in V_{\lb+1}$, we let $g_x(y)$ denote the $<_x$-smallest surjection from $V_{\lb+1}$ to $\alpha_y$, if  it  exists and  otherwise $g_x(y)=0$. 

The  map $x \mapsto g_x$ is also definable  in $L(V_{


In [71]:
#arx_id = '2310.03838v1'  # actually fails
#arx_id = "2305.15927v4"
arx_id = '2308.03822v1'
#arx_id = '2311.05999v2'
arx_id = '2302.03142v1'
arx_id = '2303.02844v1'
arx_id = '2303.15746v1'
arx_id = '2304.11351v3'
arx_id = '2305.04797v2'
arx_id = '2307.06155v5'
arx_id = '2307.07457v1'
arx_id = '2308.03822v1'
arx_id = '2309.01218v2'
arx_id = '2309.10165v2'
arx_id = '2310.14128v1'
arx_id = '2311.05966v5'
arx_id = '2311.16440v2'
arx_id = '2312.10691v1'
arx_id = '2312.13175v2'
arx_id = '2312.17319v1'
arx_id = '2303.07569v2'
arx_id = '2305.00312v3'
arx_id = '2306.01021v1'
arx_id = '2307.08203v1'
arx_id = '2307.12602v2'
arx_id = '2309.10262v2'
arx_id = '2310.18275v1'
arx_id = '2311.10270v5'
#arx_id = '2311.11770v1'
#arx_id = '2311.14619v1'  # local_def.tex
arx_id = '2310.18275v1'  ## slow
arx_id = '2308.03822v1'  # slow gemini
arx_id = '2308.03822v1'
## new ones
#arx_id = '2311.01087v2'
#arx_id = '2311.13802v2'
#arx_id = '2311.18567v2'
#arx_id = '2311.01346v2'
arx_id = '2311.00376v3'
arx_id = '2311.16333v2'
arx_id = '2311.05109v1'
#arx_id = '2311.12319v1'
arx_id = '2410.12880v2'

def append_node_contents(focus_nodes, full_nodelist, result_list, max_followers=3):
    for i,node in focus_nodes:
        temp_list = []
        temp_list.append(node.latex_verbatim())
        try:
            idx_plus = 1
            group_streak=False
            follow_count = 0
            while True:
                if idx_plus > 10:
                    break
                if follow_count > max_followers:
                    break
                if (not group_streak) and (len(temp_list) > 2):
                    break
                follow_node = full_nodelist[i+idx_plus]
                if isinstance(follow_node, LatexGroupNode):
                    temp_list.append(follow_node.latex_verbatim())
                    group_streak = True
                    follow_count += 1
                elif isinstance(follow_node, LatexCharsNode):
                    if not str(follow_node.chars).isspace():
                        group_streak = False
                else:
                    group_streak = False
                idx_plus += 1
        except IndexError:
            pass
        result_list.append("".join(temp_list))

yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"

try:
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
    debug_candidate_files, debug_include = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True, with_weights=True, file_path=tar_path)
    tex_main = candidate_files[0]
except (FileNotFoundError, ClientError, InvalidResponse, GoogleAPIError, NotFound, Forbidden) as e:
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.gz"
    tar_bytes = phase_one.bytes_from_tarpath(tar_path)
    tex_main = include_dict = None
    debug_candidate_files = debug_include = None

    
print(debug_candidate_files)

include_list = None #[]
if include_dict: include_list = include_dict.get(tex_main, [])

incl_res_gen_list = []
if include_list:
    for inc_file in include_list:
        incl_res_gen_list.append(phase_one.extract_pre_abstract_content(tar_bytes, tex_main=inc_file, file_path=tar_path, yield_sync=True))

print(f"Got main: {tex_main}")
if include_dict:
    print(f"Got included_files: {include_list}")

doc = """
Parses a .tex file:
- Removes LaTeX comments
- Extracts institution names (via recursive regex)
- Extracts text before the abstract
"""
source_text = phase_one.source_from_archive(tar_bytes, tex_main)

# Remove LaTeX comments (lines starting with non-escaped %)
new_def_str = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^}]*}\s*{[^}]*}
""".strip()
new_def_v1 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*{[^{}]*}
""".strip()
new_def_v2 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{[^{}]*}
""".strip()
new_def_v3 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength)\s*{[^{}]*}\s*(\[[^\]]*\])?\s*{(?:[^{}]*|{[^{}]*})*}\s*{(?:[^{}]*|{[^{}]*})*}
""".strip()
new_def_v4 = r"""
\\(newcommand|def|newcolumntype|renewcommand|providecommand|DeclareMathOperator|DeclareRobustCommand|newenvironment|renewenvironment|DeclareOption|newlength|newtheorem)\s*\{[^\}]+\}\s*(\[[^\]]*\])*(\s*\{\s*(\s*(?>(\\[{}]|[^{}])+|\{(?3)\})*)+\}){1,3}
""".strip()

#old # \\newenvironment\{[^\}]+\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}\s*\{\s*((?>[^{}]+|\{(?1)\})*)\}
# old \\newenvironment\s*\{[^\}]+\}\s*(\[[^\]]*\])*(\{\s*(\s*(?>[^{}]+|\{(?3)\})*)+\}){1,3}

# \\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(\s*\{\s*(\s*(?>(\\[{}]|[^{}])+|\{(?3)\})*)+\}){1,3}
strip_env = r"""
\\newenvironment\s*\{[^\}]+\}\s*(\s*\[[^\]]*\])*(?P<brgrp>\s*\{\s*(?P<inner>(?>\s+|\\\\+|\\[{}]|[^{}\\]+|\\)+|\{(?P>inner)\}+)+\s*\})+
""".strip()
#\\providecommand\{[^\}]+\}\s*(\[[^\]]*\])?\s*\{\s*((?>[^{}]+|\{(?:[^{}]*|(?1))\})*)\}
strip_provcmd = r"""
\\providecommand\{[^\}]+\}\s*(\[[^\]]*\])?\s*\{\s*((?>(\\[{}]|[^{}])+|\{(?:[^{}]*|(?1))\})*)\}
""".strip()
#new_def_v1_pat = re.compile(new_def_v1, re.DOTALL, re.MULTILINE)
#new_def_v2_pat = re.compile(new_def_v2, re.DOTALL, re.MULTILINE)
#new_def_v3_pat = re.compile(new_def_v3, re.DOTALL, re.MULTILINE)
new_def_v4_pat = re.compile(new_def_v4, re.DOTALL)
strip_env_pat  = re.compile(strip_env, re.DOTALL)
strip_provcmd_pat  = re.compile(strip_provcmd, re.DOTALL)
content = re.sub(r"(?<!\\)%.*", "", source_text)

[('acl_latex.tex', 1)]
Got main: acl_latex.tex


In [73]:
start = 0 
stop = len(content) #math.ceil(len(content)*.05)
strip_env_pat.search(content[start:stop])
print(len(content), stop)
content[stop:stop+200]

320341 320341


''

In [74]:
content = strip_provcmd_pat.sub("\n", content)
content = strip_env_pat.sub("\n", content)
content = new_def_v4_pat.sub("\n", content)
#content = new_def_v1_pat.sub("\n", content)
#content = new_def_v2_pat.sub("\n", content)
#res_list = []

In [88]:
test_max = 125
lxwkr = LatexWalker(content, tolerant_parsing=True)
(nodelist, pos, len_) = lxwkr.get_latex_nodes(read_max_nodes=test_max)
len(nodelist)
nodelist[-5:]
pos = 2228+10
width = 300
content[pos:pos+width]

125

[LatexGroupNode(parsing_state=<parsing state 140400651920448>, pos=2170, len=39, nodelist=[LatexCharsNode(parsing_state=<parsing state 140400651920448>, pos=2171, len=10, chars='skipabove='), LatexMacroNode(parsing_state=<parsing state 140400651920448>, pos=2181, len=8, macroname='topskip', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''), LatexCharsNode(parsing_state=<parsing state 140400651920448>, pos=2189, len=11, chars=',skipbelow='), LatexMacroNode(parsing_state=<parsing state 140400651920448>, pos=2200, len=8, macroname='topskip', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')], delimiters=('{', '}')),
 LatexCharsNode(parsing_state=<parsing state 140400651920448>, pos=2209, len=2, chars='\n\n'),
 LatexMacroNode(parsing_state=<parsing state 140400651920448>, pos=2211, len=11, macroname='newcounter', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space=''),
 LatexGroupNode(parsing_state=<parsing state 140400651920448>,

'\\newenvironment{theo}[1][]{\n\\stepcounter{theo}\n\\ifstrempty{#1}\n {\\mdfsetup{\n   frametitle={\n    \\tikz[baseline=(current bounding box.east),outer sep=0pt]\n    \\node[anchor=east,rectangle,fill=PeriWinkle!80]\n         {\\strut Example~\\thetheo};}}\n }\n{\\mdfsetup{\n  frametitle={\n   \\tikz[baseline=(current'

In [ ]:
# try parsing latex:
# Note: names are lowered before compare
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "university",
    "orgname",
    "affiliation", "affil", "affiliations", "aff",
    "address",
    "cmsinstitute", "icmlaffiliation",
])
supstr = set([
    "\\textsuperscript",
])
latex_extracted_institutions = []
for inc_gen in incl_res_gen_list:
    inc_res = next(inc_gen)
    if inc_res:
        latex_extracted_institutions.append(inc_res)

try:
    lxwkr = LatexWalker(content, tolerant_parsing=True)
    (nodelist, pos, len_) = lxwkr.get_latex_nodes()
    focus_nodes = [
      (i,node) for i,node in enumerate(nodelist)
      if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
    ]
    if focus_nodes:
        append_node_contents(focus_nodes, nodelist, latex_extracted_institutions)
        # Get /textsuperscript contents if indicated
        # @todo: also get the $^[1]$Institution style indicators 
        if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
            sup_res = extract_texsuperscript(nodelist)
            latex_extracted_institutions.extend(sup_res)
    else:
        doc = [
            node for node in nodelist
            if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
        ]
        if doc:
            docnodelist = doc[0].nodelist
            focus_doc_nodes = [
              (i,node) for i, node in enumerate(docnodelist)
              if isinstance(node, LatexMacroNode) and node.macroname.lower() in auth_macros
            ]
            append_node_contents(focus_doc_nodes, docnodelist, latex_extracted_institutions)
            # Get /textsuperscript contents if indicated
            # @todo: also get the $^[1]$Institution style indicators 
            if any(pat in lx for lx in latex_extracted_institutions for pat in supstr):
                sup_res = extract_texsuperscript(docnodelist)
                latex_extracted_institutions.extend(sup_res)
    #for var in ('nodelist', 'pos', 'len_', 'sup_res', 'focus_nodes', 'doc', 'docnodelist', 'focus_doc_nodes'):
    #    if var in locals(): del locals()[var]
    if latex_extracted_institutions:
        #res_list.append(latex_extracted_institutions)
        #yield "\n".join(latex_extracted_institutions)
        pass
except Exception as e:
    print(f"\nOverly broad except in extract_pre_abstract_content(): {e} for {file_path}-{tex_main}")
    pass

len(nodelist)
latex_extracted_institutions

None
Got main: None


In [68]:
focus_nodes
latex_extracted_institutions
append_node_contents(focus_nodes, nodelist, latex_extracted_institutions)
latex_extracted_institutions

[]

[]

[]

In [69]:
focus_doc_nodes
latex_extracted_institutions
append_node_contents(focus_doc_nodes, docnodelist, latex_extracted_institutions)
latex_extracted_institutions

[]

[]

[]

In [ ]:
%%time 
arx_id = '2301.07517v1'
yymm = arx_id.split(".")[0]
paper_id = arx_id.split("v")[0]
tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
tar_bytes = phase_one.bytes_from_tarpath(tar_path)

candidate_files, include_dict = phase_one.find_main_tex_source_in_tar(tar_bytes, all_found=True)
source_text = phase_one.source_from_archive(tar_bytes, candidate_files[0])
tsoup = TS.TexSoup(source_text, tolerance=0)

In [9]:
"\n".join([])

''

In [59]:
start = 2800
stop = math.ceil(len(source_text)*0.10)
print(start, stop)
content = strip_env_pat.sub("\n", source_text[start:stop])
content[start:start+20]
content[-20:]


2800 5429


''

'r, this is no longer'

In [53]:
test_ids_df = pd.read_csv("gs://institutional-extract-scratch/reference/arx_ids/2024_all_ids.csv")
time_code = "2025-05-12"
save_name = "2024_db_json"

sample_size = "all"
batch_size = 20      # articles
mp_epoch_size = 2048 #batches
parallel_workers = 28 #8
thread_workers = 10



#test_ids_df.head()
ids_2311_all = test_ids_df["arx_id"].unique()


tt = TicToc()

input_ids = set(str(x) for x in ids_2311_all)


try:
    objects = []
    with open(f"checkpoints/{save_name}_{sample_size}_{time_code}.pkl", 'rb') as cp_fp:
        while True:
            try:
                obj = pickle.load(cp_fp)
                objects.append(obj)
            except EOFError:
                break
except FileNotFoundError as e:
    pass

known_ids = []
known_res_set = set()
for obj in objects:
    known_ids.extend(str(x[0]) for x in obj)
    known_res_set.update(obj)

known_ids = set(known_ids)
input_ids = input_ids - known_ids
known_res = list(known_res_set)
print(f"Found checkpoints for {len(known_ids)} nodes.")

if True:
    arxid_itr = itr.groupby(sorted(known_res, key=lambda x: x[0]), key=lambda x: x[0])
    no_result_idx = [str(arx_id) for arx_id, grp in arxid_itr if all(x[1] in ('null', 'error') for x in grp)]
    rerun_ids = set(no_result_idx)
    print(f"Found {len(rerun_ids)} error nodes nodes to reprocess.")
    input_ids = input_ids.union(rerun_ids)

if sample_size != "all":
    input_ids = input_ids[:sample_size]

#input_ids = input_ids - skip_ids

#import concurrent.futures
#import phase_one


os.environ["TOKENIZERS_PARALLELISM"] = "false" 

mp_batch_ids = [list(input_ids)]
if len(input_ids)/batch_size > mp_epoch_size:
    #Split into epochs
    mp_batch_ids = np.array_split(list(input_ids), math.ceil(len(input_ids)/batch_size/mp_epoch_size))
mp_batches = [x for x in mp_batch_ids]
if len(input_ids) > batch_size:
    mp_batches = [
        np.array_split(x, math.ceil(len(x)/batch_size)) 
        if len(x) > batch_size
        else x 
        for x in mp_batch_ids
    ]
num_batches = sum(len(x) for x in mp_batches)

print(f"Start: {num_batches} batches in {len(mp_batches)} epochs")
tt.tic()
print(f"Start: {len(input_ids)} in {num_batches} batches")

Found checkpoints for 97540 nodes.
Found 730 error nodes nodes to reprocess.
Start: 13433 batches in 7 epochs
Start: 268572 in 13433 batches


In [11]:
batches = np.array([list(input_ids)])

In [39]:
type(batches)
type(batches[0])

list

numpy.ndarray

In [50]:
len(input_ids)/batch_size/mp_epoch_size

6.55693359375

In [40]:
if len(batches) > mp_epoch_size:
    mp_batches = np.array_split([list(x) for x in batches], math.ceil(len(batches)/mp_epoch_size))

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (13429,) + inhomogeneous part.

In [49]:
foo = np.array_split(batches[-10:], 3)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (10,) + inhomogeneous part.

In [13]:
if len(input_ids) > batch_size:
    batches = np.array_split(list(input_ids), math.ceil(len(input_ids)/batch_size))

In [14]:
type(batches)

list

In [31]:
len(batches)
math.ceil(len(batches)/mp_epoch_size)

13429

7

In [26]:
np.array_split(batches, 7)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (13429,) + inhomogeneous part.